In [ ]:
import pandas as pd
from ortools.sat.python import cp_model
import random
from datetime import datetime
import time

In [ ]:
people_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main People')
jobs_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main Desk')

In [ ]:
# map all the value 
grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'F':5}
edu_map = {'Doc': 1, 'Degree':2, 'Uni':2, 'Dipolma':3, 'O Level':4, 'N Level':5}
health_map = {'Fit':1, 'Semi Fit':2, 'Not Fit': 3}
sec_map = {'CAT1':1, 'CAT2':2, 'CAT3':3, 'CAT4':4, 'CAT5':5, 'CAT6':6, 'CAT7':7, 'CAT8':8, 'CAT9':9, 'CAT10':10}

jobs_df.info()

In [ ]:
people_df['Grade'] = people_df['Grade'].replace(grade_map)
people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
people_df['Health'] = people_df['Health'].replace(health_map)
people_df['Security'] = people_df['Security'].replace(sec_map)

jobs_df['Req_Grade'] = jobs_df['Req_Grade'].replace(grade_map)
jobs_df['Req_Edu'] = jobs_df['Req_Edu'].replace(edu_map)
jobs_df['Req_Health'] = jobs_df['Req_Health'].replace(health_map)
jobs_df['Sec_Clerance'] = jobs_df['Sec_Clerance'].replace(sec_map)

In [ ]:
appointment_cols = ["Appointments_1", "Appointments_2", "Appointments_3", "Appointments _4"]

# Get all the appointment and put into 1 column
people_df["Appointments"] = (
    people_df[appointment_cols]
    .apply(lambda row: [a for a in row if pd.notna(a) and a != ""], axis=1)
)
people_df = people_df.drop(columns=appointment_cols)

people_df

In [ ]:
def str_time_prop(start, end, time_format, prop):
    stime = time.mktime(time.strptime(start, time_format))
    etime = time.mktime(time.strptime(end, time_format))
    ptime = stime + prop * (etime - stime)
    return time.strftime(time_format, time.localtime(ptime))

def random_date(start, end, prop):
    return str_time_prop(start, end, '%d/%m/%Y', prop)

def start_date():
    return random_date(f"1/1/{2021}", f"1/1/{2024}", random.random())

def end_date(start_date):
    start_year=start_date[-4:]
    end_start, end_end = int(start_year) +2, int(start_year) +5
    return random_date(f"1/1/{end_start}", f"1/1/{end_end}", random.random())    

In [ ]:
people_df["Start_Date"] = people_df.apply(lambda _: start_date(), axis=1)
people_df["End_Date"] = people_df.apply(lambda _: end_date(start_date()), axis=1)

people_df

In [ ]:
people_df["End_Date"] = pd.to_datetime(people_df["End_Date"])
today = pd.Timestamp.today().normalize()
people_df["Month_to_Post_Out"] = (
    (people_df["End_Date"].dt.year - today.year) * 12
    + (today.month - people_df["End_Date"].dt.month)
)

In [ ]:
people_df["overdue"] = people_df["month_diff"] > 36
people_df

In [ ]:
# work on the model (Unless I want to give classification on the appointment ie IT, Mgmt etc) 

In [ ]:
print("Starting the model")

In [ ]:
model = cp_model.CpModel()
assign = {}

# Getting all people who matches the job base on hard constraints
for p_idx, person in people_df.iterrows():
    for j_idx, job in jobs_df.iterrows():
        if person['Health'] != job['Req_Health']:
            continue
        if person['Security'] != job['Sec_Clerance']:
            continue
        var = model.NewBoolVar(f"assign_p{p_idx}_j{j_idx}")
        assign[(p_idx, j_idx)] = var


In [ ]:
#Soft constraints
appointment_terms = []
edu_terms = []

for (p_idx, j_idx), var in assign.items():
    person_apps = people_df.loc[p_idx, "Appointments"]
    job_app = jobs_df.loc[j_idx, "Req_Appointment"]
    
    if job_app in person_apps:
        app_idx = person_apps.index(job_app)
        appoint_score = len(person_apps) - app_idx
    else:
        appoint_score = 0
    appointment_terms.append(var * appoint_score * 5)  # weight (higher the better)

    edu_score = 1 if people_df.loc[p_idx, "Edu_Type"] == jobs_df.loc[j_idx, "Req_Edu"] else 0
    edu_terms.append(var * edu_score * 1) 

In [ ]:
model.Maximize(sum(appointment_terms) + sum(edu_terms) )

solver = cp_model.CpSolver()
status = solver.Solve(model)

In [ ]:
results = []

for (p_idx, j_idx), var in assign.items():
    if solver.BooleanValue(var):
        appoint_match = (
            jobs_df.loc[j_idx, "Req_Appointment"]
            in people_df.loc[p_idx, "Appointments"]
        )
        edu_match = (
            people_df.loc[p_idx, "Edu_Type"]
            == jobs_df.loc[j_idx, "Req_Edu"]
        )

        results.append({
            "Person": people_df.loc[p_idx, "Name"],
            "Job": jobs_df.loc[j_idx, "Desk_ID"],
            "Appointment_Match": int(appoint_match),
            "Edu_Match": int(edu_match),
            "Suitable": int(appoint_match) + int(edu_match)
        })

result_df = pd.DataFrame(results)
print(result_df)

In [ ]:
suitable_df = result_df.loc[result_df['Suitable'] == 2]
suitable_df

In [ ]:
result_df.to_excel("Result.xlsx")